# Docker Compose & Multi-Container Apps

## What's covered

- **Why Compose** — `docker run` doesn't scale past one container, and bash scripts are not the answer
- **The `compose.yaml` shape** — `services`, `networks`, `volumes`, `configs`, `secrets`, `name`
- **A first compose file** — web + db, end to end
- **The everyday commands** — `up`, `down`, `ps`, `logs`, `exec`, `build`, `restart`, `pull`, `config`
- **`build:` vs `image:`** — when to build inline, when to reference a pre-built image
- **`depends_on` with healthchecks** — startup order that actually works
- **Networks and volumes in Compose** — the implicit defaults vs explicit declarations
- **Environment** — three layers: image `ENV`, `environment:`, `env_file:`, plus `.env` for substitution
- **Profiles** — services that opt in to specific use cases (dev, test, debug)
- **Overrides** — `compose.override.yaml`, `-f` stacking, per-environment layering
- **Project name** — what `docker compose -p NAME` actually changes, and why two checkouts of the same repo collide
- **Scaling** — `--scale web=3`, the load balancer, and what *doesn't* scale this way
- **A realistic stack** — Flask + Postgres + Redis, with healthchecks, secrets via env, persistent data, and the right startup order

By the end you should be able to declare any multi-container app in a single YAML file, bring it up and down with one command, and explain every default Compose silently applied.

## Why Compose

Notebook 03 had you typing `docker run` with seven flags to launch a single container. Now imagine a normal web app: a web server, a database, a Redis cache, maybe a worker. That's four containers, four `docker run` lines, and:

- A user-defined network that connects them
- Two named volumes (Postgres data, Redis dump)
- Environment variables shared by web and worker
- A specific startup order — DB must be up before web
- Port mappings — only web should be exposed
- The same flags repeated across `docker run`s

You can do this with a bash script. Teams have. It's miserable to maintain.

**Docker Compose** is the answer. One YAML file (`compose.yaml`) describes the whole stack. `docker compose up` brings everything up; `docker compose down` tears it down. The file is checked into git; teammates clone and run.

Compose isn't an orchestrator (that's notebook 09 — Swarm — or Kubernetes). It's a *local development and single-host deployment* tool. It runs N containers, on one Docker daemon, declaratively.

### Compose v2 vs Compose v1

Two things called "Compose" exist:

- **Compose v1** — the old standalone `docker-compose` Python binary (hyphenated). End-of-life, no new features.
- **Compose v2** — the modern `docker compose` subcommand (space, not hyphen), shipped as a Go plugin to the Docker CLI. This is what you use today.

Both read the same YAML. The differences are minor: v2 is faster, supports newer schema features, and integrates with `docker context`. **Use v2 (`docker compose`)** and never look back.

## The `compose.yaml` shape

The conventional filename is `compose.yaml` (`docker-compose.yml` and `docker-compose.yaml` also work for backward compatibility). The minimum-viable file:

```yaml
services:
  web:
    image: nginx:alpine
```

That's a valid Compose file: one service called `web`, runs `nginx:alpine`. `docker compose up` brings it up. Docker auto-creates a network for the project and joins `web` to it.

The full top-level structure:

```yaml
name: my-app                # project name (optional; default = directory name)

services:                   # the containers
  web:
    image: ...
    build: ...
    ports: ...
    environment: ...
    volumes: ...
    depends_on: ...
    networks: ...
    healthcheck: ...
    restart: ...
    # ... and ~50 more keys

networks:                   # named networks (optional; one default is auto-created)
  backend:
    driver: bridge

volumes:                    # named volumes (optional; like `docker volume create`)
  pgdata: {}

configs:                    # config files mounted into services (used more in Swarm)
  nginx_conf:
    file: ./nginx.conf

secrets:                    # secrets (also Swarm-flavoured, but works in Compose)
  db_password:
    file: ./secrets/db_password.txt
```

We'll meet each block in turn.

### What Compose does silently

When you `docker compose up`, Compose:

1. Picks a **project name** (the directory name, unless you override).
2. Creates a **default network** called `<project>_default` (driver `bridge`) and attaches every service that doesn't specify its own networks.
3. Creates any named volumes and networks declared in the file.
4. Starts services in **dependency order** (`depends_on`).
5. Tags each container with labels so it knows they belong to this project.

That last point is how `docker compose down` knows what to clean up — it filters by the project label.

## A first compose file — web + db

```yaml
# compose.yaml
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: secret
      POSTGRES_DB: app
    volumes:
      - pgdata:/var/lib/postgresql/data

  web:
    image: nginx:alpine
    ports:
      - "8080:80"
    depends_on:
      - db

volumes:
  pgdata: {}
```

Three things to notice:

1. **No explicit network.** Both services land on the default network (`<project>_default`). `web` can reach `db` by hostname `db` automatically — the same embedded-DNS magic from notebook 05.
2. **Named volume.** `pgdata` is declared under top-level `volumes:` and referenced in `db`'s `volumes:`. Compose creates it on first `up`.
3. **`depends_on: db`.** Compose starts `db` first, then `web`. (We'll see why this isn't *quite* enough on its own in the next section.)

Bring it up:

```bash
docker compose up -d        # detached
docker compose ps           # see what's running
docker compose logs -f web  # tail web logs
docker compose down         # stop & remove (volumes survive)
docker compose down -v      # also wipe named volumes
```

## The everyday commands

| Command | What it does |
|---|---|
| `docker compose up [-d]` | Create + start every service. `-d` for detached. |
| `docker compose up -d --build` | Rebuild images before starting (for `build:` services). |
| `docker compose down [-v] [--rmi all]` | Stop and remove containers and the project's network. `-v` removes named volumes; `--rmi all` removes images. |
| `docker compose ps` | List the project's containers and their state. |
| `docker compose logs [-f] [SERVICE...]` | Tail logs. `-f` to follow. Specify a service or get all. |
| `docker compose exec SERVICE CMD` | Run a command inside a running service. The Compose equivalent of `docker exec`. |
| `docker compose run --rm SERVICE CMD` | Spin up a **new** container of `SERVICE` to run `CMD` once. Useful for one-shot tasks (migrations). |
| `docker compose build [SERVICE]` | Build (or rebuild) images for services with `build:`. |
| `docker compose pull` | Pull the latest images for services with `image:`. |
| `docker compose restart [SERVICE]` | Restart a service without re-creating the container. |
| `docker compose stop / start` | Stop/start without removing. |
| `docker compose config` | Print the fully-resolved YAML (after variable substitution and overrides). The first thing to run when "it isn't doing what I expected." |
| `docker compose top` | Like `docker top` but for the whole stack. |

The two you'll use most: `up -d`, `down`, and `logs -f`. Followed closely by `exec` (for debugging) and `config` (when something's mysterious).

## `build:` vs `image:`

Each service must specify what to run, with one of two keys.

**`image:`** — run a pre-built image. Pulled from a registry if not local.

```yaml
services:
  cache:
    image: redis:7-alpine
```

**`build:`** — build an image from a Dockerfile in the project. The shortest form:

```yaml
services:
  web:
    build: .
```

`build: .` means "the build context is `.` and the Dockerfile is `./Dockerfile`." Long form gives you more control:

```yaml
services:
  web:
    build:
      context: ./web
      dockerfile: Dockerfile.prod
      args:
        APP_VERSION: 1.2.3
      target: production    # multi-stage build target
    image: my-org/web:1.2.3  # tag for the built image
```

You can **specify both `build:` and `image:`** — Compose builds, then tags with `image:`. That tag is what gets pushed if you `docker compose push`.

`docker compose up` builds images that don't exist locally. To force a rebuild, `docker compose up --build` or explicit `docker compose build`.

## `depends_on` with healthchecks — startup order that actually works

`depends_on: [db]` is doing less than people think.

**Default behaviour** — Compose starts `db` *before* `web`. But `db`'s **process** is up after a millisecond; the **Postgres server** inside it isn't accepting connections for another second or two. `web` may start, try to connect, and crash before `db` is ready. The `depends_on` only waits for "the container has started," not "the service inside it is ready."

The fix is `depends_on` with a **condition**, coupled with a **healthcheck**:

```yaml
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: secret
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres"]
      interval: 5s
      timeout: 3s
      retries: 5
      start_period: 10s

  web:
    image: myorg/web
    depends_on:
      db:
        condition: service_healthy
```

Now Compose starts `db`, polls its healthcheck, and only starts `web` once the healthcheck reports `healthy`. The four conditions you'll see:

- **`service_started`** — wait for container start (the old default).
- **`service_healthy`** — wait for healthcheck to pass.
- **`service_completed_successfully`** — wait for a one-shot service (a migrator container) to exit with code 0.

This pattern — "depend on `service_healthy`" — is the standard production-quality Compose idiom. Memorise it.

## Networks and volumes in Compose

You can use Compose for years without ever declaring an explicit network — the default one is fine. But there are two reasons you'd add explicit networks.

**Tier separation.** A web app where the reverse proxy can see the API, the API can see the DB, but the proxy can't see the DB directly:

```yaml
services:
  proxy:
    networks: [frontend]
  api:
    networks: [frontend, backend]
  db:
    networks: [backend]

networks:
  frontend: {}
  backend: {}
```

**External networks.** Connect to a network created outside this Compose file (e.g. by another Compose project, or by `docker network create`):

```yaml
networks:
  shared:
    external: true
    name: shared-net
```

For **volumes**, the pattern from notebook 04 carries over. Declare named volumes under top-level `volumes:` and reference them in the service. The advantage in Compose: the volume's lifecycle is tied to the project. `docker compose down -v` removes it; without `-v` it persists across `up`/`down` cycles.

Bind mounts use the host path directly. **Compose resolves relative paths against the directory of the `compose.yaml` file**, which is why `./src:/app/src` works regardless of where you ran `docker compose up` from.

## Environment — three layers and an `.env` file

The three places service-level env vars come from, in precedence order (lowest to highest):

1. **Image `ENV`** — baked into the Dockerfile.
2. **`env_file:`** — one or more files of `KEY=value` lines.
3. **`environment:`** — explicit map or list in the YAML. Wins.

```yaml
services:
  api:
    image: myorg/api
    env_file:
      - ./api.env
    environment:
      LOG_LEVEL: debug         # overrides anything in api.env
      DATABASE_URL: postgres://app:secret@db:5432/app
```

### The `.env` file (variable substitution)

Compose **also** reads a file called `.env` next to `compose.yaml`, but for a different purpose: **substituting variables into the YAML itself**.

```
# .env
TAG=1.2.3
DB_PASSWORD=local-only
```

```yaml
services:
  api:
    image: myorg/api:${TAG}            # becomes myorg/api:1.2.3
    environment:
      DB_PASSWORD: ${DB_PASSWORD}      # passes the value to the container
```

**The distinction:**

- `.env` → variables for the YAML file's text (compile-time, at parse).
- `env_file:` → variables for the *container's* environment (runtime, inside the service).

They're easy to confuse. Read the keyword carefully.

### Defaults and required values

In substitutions, you can default and require:

```yaml
api:
  image: myorg/api:${TAG:-latest}    # default to "latest"
  environment:
    SENTRY_DSN: ${SENTRY_DSN:?must be set}   # error if unset
```

## Profiles — opt-in services

Sometimes a service should only run in specific contexts — a debug sidecar, a seed-data container, a load test. **Profiles** let you tag services so they're skipped unless you explicitly enable the profile.

```yaml
services:
  web:
    image: myorg/web

  debugger:
    image: nicolaka/netshoot
    profiles: [debug]
    command: sleep infinity

  seed:
    image: myorg/db-seed
    profiles: [seed]
```

- `docker compose up` — starts only `web` (the no-profile default).
- `docker compose --profile debug up` — starts `web` *and* `debugger`.
- `docker compose --profile debug --profile seed up` — all three.

A service with **no** `profiles:` always runs. A service with a `profiles:` list runs only when at least one of its profiles is enabled. This is great for stacks where a few services are dev/test-only and shouldn't pollute the default `up`.

## Overrides — layering compose files

You'll often want different settings for dev vs CI vs prod. Compose has two mechanisms.

### Automatic `compose.override.yaml`

If a file called `compose.override.yaml` (or `.yml`) sits next to `compose.yaml`, Compose loads **both** and merges. The override wins on conflicts.

```yaml
# compose.yaml — base
services:
  web:
    image: myorg/web
    restart: unless-stopped

# compose.override.yaml — dev tweaks (auto-loaded locally, gitignored)
services:
  web:
    build: .
    volumes:
      - ./web:/app
    environment:
      DEBUG: "1"
```

Convention: `compose.yaml` is the production-ish base, `compose.override.yaml` is dev-only and gitignored. New devs clone and `up` and get the dev-friendly stack.

### Explicit `-f` stacking

For more layers, pass `-f` multiple times. Later files override earlier ones.

```bash
docker compose -f compose.yaml -f compose.prod.yaml up
docker compose -f compose.yaml -f compose.ci.yaml up
```

This pattern beats environment-variable spaghetti for any non-trivial deploy.

### How merging works

- Scalars (like `image:`, `restart:`) — override replaces base.
- Lists (like `ports:`, `volumes:`, `environment:` when written as a list) — *append*, not replace. Watch for unexpected duplicates.
- Maps (like `environment:` when written as a map, `labels:`) — merge key by key.

When the merge result is unclear, **`docker compose -f a.yaml -f b.yaml config`** prints the resolved YAML. Use it.

## Project name — what it means and why duplicates collide

Compose namespaces everything it creates by a **project name**:

- Containers: `<project>-<service>-<index>` (e.g. `my-app-web-1`)
- Networks: `<project>_default` (or `<project>_<network-name>`)
- Volumes: `<project>_<volume-name>`
- A label `com.docker.compose.project=<project>` on every object

Default is the **basename of the directory** containing `compose.yaml`. Override with:

- `name: my-app` at the top of `compose.yaml`
- `docker compose -p my-app up`
- `COMPOSE_PROJECT_NAME=my-app` in the environment / `.env`

### Why two checkouts of the same repo collide

If you clone the same project into `~/work/myapp` and `~/scratch/myapp` and `docker compose up` in each, **they'll fight over container names and volumes** because both default the project name to `myapp`. The second one will error or steal the first's containers.

Fix: pick a unique project name per checkout. The simplest is to rename one directory; the explicit fix is `docker compose -p myapp-scratch up` (or put `COMPOSE_PROJECT_NAME=myapp-scratch` in `~/scratch/myapp/.env`).

## Scaling — and what it does (and doesn't) give you

```bash
docker compose up -d --scale web=3
```

Start three containers of the `web` service. Compose names them `myapp-web-1`, `myapp-web-2`, `myapp-web-3`. They all join the project network, all answer to the DNS name `web` (the embedded resolver round-robins between their IPs).

**What this is good for:**

- Local load testing.
- Horizontally-scalable stateless services (the canonical "web" tier).

**What it is *not* good for:**

- **Port mappings.** If the service has `ports: ["8080:80"]`, **only one replica can bind 8080**. The others fail. To scale a published service, drop the host-port pin and put a load balancer in front (Nginx, Traefik) or use a random port range like `"8080-8082:80"`.
- **Stateful services.** Don't `--scale` Postgres. You'll get N Postgres containers fighting over the same data dir.
- **Production deployments across hosts.** That's Swarm or Kubernetes — single-host Compose can only scale to the limits of one machine.

## A realistic stack — Flask + Postgres + Redis, end to end

Bringing it all together. The same Flask app from notebook 02, but now wired to a Postgres for storage and a Redis for caching.

```yaml
# compose.yaml
name: rt-demo

services:
  db:
    image: postgres:16
    environment:
      POSTGRES_USER: app
      POSTGRES_PASSWORD: ${DB_PASSWORD:?DB_PASSWORD must be set}
      POSTGRES_DB: app
    volumes:
      - pgdata:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U app -d app"]
      interval: 5s
      timeout: 3s
      retries: 5
      start_period: 10s
    restart: unless-stopped

  cache:
    image: redis:7-alpine
    volumes:
      - redisdata:/data
    healthcheck:
      test: ["CMD", "redis-cli", "PING"]
      interval: 5s
      timeout: 3s
      retries: 5
    restart: unless-stopped

  web:
    build: ./web
    ports:
      - "127.0.0.1:8080:8080"      # bind to localhost only
    environment:
      DATABASE_URL: postgres://app:${DB_PASSWORD}@db:5432/app
      REDIS_URL: redis://cache:6379/0
      LOG_LEVEL: info
    depends_on:
      db:
        condition: service_healthy
      cache:
        condition: service_healthy
    restart: unless-stopped

volumes:
  pgdata: {}
  redisdata: {}
```

```
# .env (gitignored)
DB_PASSWORD=local-only-secret
```

What this stack gets right:

- **DB password is required.** `${DB_PASSWORD:?...}` errors if missing — no accidental empty-password deploys.
- **`web` waits for both `db` and `cache` to be healthy.** No flaky startup races.
- **Port is bound to `127.0.0.1`.** The app isn't accidentally exposed on the host's external interface.
- **All three services restart.** A crashed container comes back automatically (and survives daemon restarts).
- **Volumes are named.** Data persists across `down`/`up`. `down -v` is what nukes data.
- **Project name pinned via `name:`.** No collisions with other checkouts.

### Day-to-day commands

```bash
docker compose up -d                      # bring everything up
docker compose ps                         # status, including health
docker compose logs -f web                # tail just the web logs
docker compose exec web sh                # drop into the web container
docker compose exec db psql -U app app    # psql into the DB
docker compose run --rm web flask db upgrade   # one-shot migration

docker compose pull && docker compose up -d   # update images, recreate changed services
docker compose down                       # stop & remove (data survives)
docker compose down -v                    # also wipe data — careful
```

This pattern — declarative stack, healthchecked dependencies, named volumes, bound localhost ports, restart policy on everything — is what a Compose-based local dev or single-host deploy should look like in 2026.